In [1]:
# Import packages
from dotenv import load_dotenv
from anthropic import Anthropic
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
# Set up LLM model and API key
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY2')
model_name = "claude-opus-4-6"

In [3]:
# Initialize the Anthropic client
claude = Anthropic()

## Input artifacts

In [4]:
# Read the PDF file and extract text
reader = PdfReader("3_lab3_daniel/profile.pdf")
document = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        document += text
# print(document)

In [5]:
# Read the summary from the text file
with open("3_lab3_daniel/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
# Assign the name of the person to the system prompt
name = "Daniel"

In [7]:
system_prompt = f"You are acting as {name}. You are a senior AI Agentic Engineer. You will provide support and insghts for AI, Cyber security."

system_prompt += f"\n\n## Document:\n{document}\n\n"
system_prompt += f"\n\n## Summary:\n{summary}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}. Be short, concise, and provide actionable insights."
system_prompt += f"\n\n If you don't know the answer, say 'I don't know' instead of making up an answer."


## One Shot

In [8]:
def chat(message, history):
    # return f"You said:  {message}" Test
    # Gradio adds extra keys (e.g. "metadata") to history dicts; Claude's API rejects unknown keys, so strip down to role/content
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # Append the new user turn to the cleaned conversation history
    messages = history + [{"role": "user", "content": message}]
    # Send the full conversation + persona system prompt to Claude
    response = claude.messages.create(
        model=model_name, 
        messages=messages,
        timeout=60,
        max_tokens=1024,
        system=system_prompt
        )
    return next(block.text for block in response.content if block.type == "text")

In [ ]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

## Tools

LLMs, tools, and agents boil down to functions, JSON, and conditional logic — wrapped around a probabilistic reasoner (the LLM) that decides which function to call, with what arguments, and when.

In [8]:
# Define the 'record_email' tool function
def record_email(email):
    with open("3_lab3_daniel/emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email recorded successfully."

# Specifying Client tools
record_email_tool = {
    "name": "record_email", # Link to function
    "description": "Record a user's email address to a text file. Call this whenever the user provides their email address during the conversation.",
    "input_schema": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The email address of the user to record",}
            },
            "required": ["email"],
        },
}

In [9]:
# Define the 'record_interest' tool function
def record_interest(interest_str):
    with open("3_lab3_daniel/interests.txt", "a", encoding="utf-8") as f:
        f.write(interest_str + "\n")
    return "Interest recorded successfully."

# Specifying Client tools
record_interest_tool = {
    "name": "record_interest", # Link to function
    "description": "Based in the conversation, record a user's interests. Consider one round as one reply and its response, bilateral. Do it each time.",
    "input_schema": {
            "type": "object",
            "properties": {
                "interest": {"type": "string", "description": "The interest of the user to record",}
            },
            "required": ["interest"],
        },
}

In [10]:
# Dispatcher  — canonical, works for 1 or N tools
def run_tool(name, tool_input):
    if name == "record_email":
        return record_email(tool_input["email"])
    if name == "record_interest":
        return record_interest(tool_input["interest"])
    raise ValueError(f"Unknown tool: {name}")

In [13]:
# Tools list
tools = [record_email_tool, record_interest_tool]

In [17]:
# Chat function (Gradio) with tool support
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history] # Keep only role, content. Drop Gradio's extra keys
    messages = history + [{"role": "user", "content": message}] # Append the user turn

    while True:
        response = claude.messages.create(
            model=model_name,
            messages=messages,
            timeout=59,
            max_tokens=1024,
            system=system_prompt,
            tools=tools, # List of dictionaries at disposal of the LLM
        )
        if response.stop_reason != "tool_use": # Checks whethet Claude is done (not requesting a tool)
            return next(b.text for b in response.content if b.type == "text") # Return the text of the first block

        messages.append({"role": "assistant", "content": response.content}) # Record Claude's turn (text + tool_use blocks) in the conversation

        tool_results = [] # Collect one tool_result per tool_use block below
        for block in response.content:  # Scan every block in Claude's reply (text and tool_use)
            print(block)
            if block.type == 'tool_use':
                result = run_tool(block.name, block.input) # block.name, block.input are set by Claude
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})  # Send all tool results back as one user message

In [ ]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


TextBlock(citations=None, text='\n\nThanks, John! Let me save your info real quick.', type='text')
ToolUseBlock(id='toolu_01VEJrWdVyircwdqPJ8RTboK', input={'email': 'john123@gmail.com'}, name='record_email', type='tool_use', caller={'type': 'direct'})
ToolUseBlock(id='toolu_01Sm6znfMQ5ZndvSt84NLT8K', input={'interest': 'Interested in Daniel writing to them / potential collaboration or communication'}, name='record_interest', type='tool_use', caller={'type': 'direct'})
ToolUseBlock(id='toolu_01CWyqwP3tVFV3rWipQK2af1', input={'email': 'john123@gmail.com'}, name='record_email', type='tool_use', caller={'type': 'direct'})
ToolUseBlock(id='toolu_01UJcuAwvzYkQhDT7KiY1tYL', input={'interest': 'coffee'}, name='record_interest', type='tool_use', caller={'type': 'direct'})
ToolUseBlock(id='toolu_01S2ts5JfuGUtH3DvQEyW7jU', input={'interest': 'Background experience and career history'}, name='record_interest', type='tool_use', caller={'type': 'direct'})
ToolUseBlock(id='toolu_01WeX2w7QGTPVwLYpRBAL

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [ ]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()